In [2]:
import os, re, cv2, torch, pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
from torchvision.transforms import ToTensor
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import random
import matplotlib.pyplot as plt


train_folder = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\AND_result_fold2")
test_folder  = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\AND_result_test")
label_path   = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\scientificProject\data\labels.csv"

label_df = pd.read_csv(label_path)
label_df["name"] = label_df["name"].astype(str)
label = {int(row["name"].split('.')[0]): row["grade"] for _, row in label_df.iterrows()}

def get_id(filename: str) -> int:
    return int(re.findall(r'\d+', Path(filename).stem)[0])

class CustomImageDataset(Dataset):
    def __init__(self, image_folder, labels, transform=None):
        self.image_folder = image_folder
        self.image_files = sorted(
            [f for f in os.listdir(image_folder) if f.lower().endswith(('.jpg', '.png'))],
            key=get_id
        )
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = self.image_folder / img_name
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image)

        img_id = get_id(img_name)
        label = torch.tensor(self.labels.get(img_id, 0), dtype=torch.long) 
        return image, label

# تبدیل‌ها
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# ساخت دیتاست
train_dataset = CustomImageDataset(train_folder, label, transform=transform)
test_dataset  = CustomImageDataset(test_folder, label, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)                     # Python random
    np.random.seed(seed)                  # NumPy random
    torch.manual_seed(seed)               # PyTorch CPU
    torch.cuda.manual_seed(seed)          # PyTorch GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
set_seed(42)

In [12]:
# def morphological_transform(image):
#     kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
#     # باز کردن (erosion بعد dilation)
#     opened = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel)
#     return opened
# 
# class CustomImageDataset(Dataset):
#     def __init__(self, image_folder, labels, transform=None):
#         self.image_folder = image_folder
#         self.image_files = sorted(
#             [f for f in os.listdir(image_folder) if f.lower().endswith(('.jpg', '.png'))],
#             key=get_id
#         )
#         self.labels = labels
#         self.transform = transform
# 
#     def morphological_transform(self, image):
#         kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
#         image = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel)
#         # اگر خواستی می‌تونی تو اینجا morph های دیگه هم اضافه کنی
#         return image
# 
#     def __len__(self):
#         return len(self.image_files)
# 
#     def __getitem__(self, idx):
#         img_name = self.image_files[idx]
#         img_path = self.image_folder / img_name
#         image = cv2.imread(str(img_path))
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
# 
#         # اعمال فیلتر مورفولوژیکی
#         image = self.morphological_transform(image)
# 
#         if self.transform:
#             image = self.transform(image)
# 
#         img_id = get_id(img_name)
#         label = torch.tensor(self.labels.get(img_id, 0), dtype=torch.long) 
#         return image, label

class CustomImageDataset(Dataset):
    def __init__(self, image_folder, labels, transform=None):
        self.image_folder = image_folder
        self.image_files = sorted(
            [f for f in os.listdir(image_folder) if f.lower().endswith(('.jpg', '.png'))],
            key=get_id
        )
        self.labels = labels
        self.transform = transform

    def heatmap_transform(self, image):
        # تبدیل به grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        # نرمالایز 0 تا 255
        norm = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        # اعمال colormap (مثلاً Jet)
        heatmap = cv2.applyColorMap(norm, cv2.COLORMAP_JET)
        # باز تبدیل به RGB چون applyColorMap BGR میده
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        return heatmap

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = self.image_folder / img_name
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # اعمال heatmap روی تصویر
        image = self.heatmap_transform(image)

        if self.transform:
            image = self.transform(image)

        img_id = get_id(img_name)
        label = torch.tensor(self.labels.get(img_id, 0), dtype=torch.long)
        return image, label


In [15]:
class CNN_MLP(nn.Module):
    def __init__(self, num_classes):
        super(CNN_MLP, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  
            nn.ReLU(),
            nn.MaxPool2d(2),  
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  
            nn.ReLU(),
            nn.MaxPool2d(2),  
        )
        self.flatten = nn.Flatten()
        self.mlp = nn.Sequential(
            nn.Linear(64 * 32 * 32, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)
        x = self.flatten(x)
        x = self.mlp(x)
        return x


In [16]:
def compute_metrics(y_true, y_pred, average='macro'):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average=average)
    recall = recall_score(y_true, y_pred, average=None)  # sensitivity per class
    cm = confusion_matrix(y_true, y_pred)
    
    specificity = []
    for i in range(len(cm)):
        tn = np.sum(np.delete(np.delete(cm, i, axis=0), i, axis=1))
        fp = np.sum(np.delete(cm, i, axis=0)[:, i])
        specificity.append(tn / (tn + fp + 1e-10))
    
    spec = np.mean(specificity)
    sens = np.mean(recall)

    return acc, f1, spec, sens, cm

num_classes = len(set(label.values()))
model = CNN_MLP(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
num_epochs = 10

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    y_true_train, y_pred_train = [], []

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        y_true_train.extend(labels.cpu().numpy())
        y_pred_train.extend(predicted.cpu().numpy())

    epoch_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)

    acc_train, f1_train, spec_train, sens_train, _ = compute_metrics(y_true_train, y_pred_train)

    # ---- محاسبه val_loss و ارزیابی روی داده‌های تست ---- #
    model.eval()
    y_true_test, y_pred_test = [], []
    val_running_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            y_true_test.extend(labels.cpu().numpy())
            y_pred_test.extend(predicted.cpu().numpy())

    val_loss = val_running_loss / len(test_loader.dataset)
    val_losses.append(val_loss)

    acc_test, f1_test, spec_test, sens_test, _ = compute_metrics(y_true_test, y_pred_test)

    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"Train → accuracy: {acc_train:.4f}  f1 score: {f1_train:.4f}  spec: {spec_train:.4f}  sens: {sens_train:.4f}")
    print(f"Test  → accuracy: {acc_test:.4f}  f1 score: {f1_test:.4f}  spec: {spec_test:.4f}  sens: {sens_test:.4f}")



In [14]:
plt.figure(figsize=(10,6))
plt.plot(train_losses, label="Train Loss", marker='o')
plt.plot(val_losses, label="Validation Loss", marker='s')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

In [17]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# تابع تعریف مدل با هایپرپارامتر dropout قابل تغییر
class CNN_MLP(nn.Module):
    def __init__(self, num_classes, dropout_rate):
        super(CNN_MLP, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.flatten = nn.Flatten()
        self.mlp = nn.Sequential(
            nn.Linear(64 * 32 * 32, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)
        x = self.flatten(x)
        x = self.mlp(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    # هایپرپارامترهای قابل تنظیم
    lr = trial.suggest_loguniform('lr', 1e-5, 1e-2)
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32])
    dropout = trial.suggest_float('dropout', 0.1, 0.5)

    # ساخت دیتالودر با batch size پیشنهادی
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = CNN_MLP(num_classes, dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    num_epochs = 5  # برای تیونینگ کوتاه

    for epoch in range(num_epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # ارزیابی روی داده تست
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    return acc  # می‌خوایم accuracy رو maximize کنیم

# اجرای تیونینگ با Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print("Best trial:")
trial = study.best_trial
print(f"  Accuracy: {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")
